# Demo LIVE 2 · Senza guardare i polmoni

**CAF GIAS, Università degli Studi Link, modulo 2.4** · Michele Ferramola · slide «Senza guardare i polmoni»

**Il messaggio.** Un buon AUROC non dice che cosa il modello ha imparato. Qui trova l'edema polmonare
senza vedere i polmoni, perché la lastra fatta a letto si fa ai pazienti più gravi.

**Prima di eseguire, chiedi.** Senza vedere l'immagine, sapendo solo come è stata fatta la lastra,
un modello può riconoscere l'edema? Se non c'è nessun indizio, AUROC 0,5.

**Come si usa.** Circa 4 minuti. Una cella alla volta, con Maiusc + Invio. Sotto ogni passo c'è il numero atteso.

**I dati.** NIH ChestX-ray14 (NIH Clinical Center): 112.120 radiografie del torace di 30.805 pazienti.
Usiamo solo i metadati, nessuna immagine. I reperti sono estratti in automatico dai referti, quindi contengono errori.
Uso libero citando Wang X. e colleghi, *ChestX-ray8*, IEEE CVPR 2017. Dati originali: https://nihcc.app.box.com/v/ChestXray-NIHCC

### Passo 1 · Di ogni radiografia teniamo età, sesso, proiezione e apparecchio

`a_letto` vale 1 se la lastra è fatta a letto (proiezione AP), 0 se in piedi (PA).
Larghezza, altezza e dimensione dei pixel identificano l'apparecchio.

**Atteso:** 112.120 radiografie, 30.805 pazienti, il 40% fatte a letto.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

DATI = "https://raw.githubusercontent.com/MicheleFerramola00/lezioni/caf-unilink-m2.4/dati/"
d = pd.read_parquet(DATI + "radiografie_nih_metadati.parquet")
print(f"{len(d):_} radiografie, {d.paziente.nunique():_} pazienti, fatte a letto: {d.a_letto.mean():.0%}".replace("_", "."))
d[["eta", "donna", "a_letto", "larghezza", "altezza", "edema"]].head()

### Passo 2 · Chi fa la lastra a letto

La lastra a letto si fa a chi non può alzarsi: i pazienti più gravi.

**Atteso:** a letto l'88% delle radiografie con edema, il 35% di quelle senza reperti.

In [ ]:
reperti = ["nessun_reperto", "versamento", "polmonite", "consolidamento", "edema"]
quota = pd.Series({r.replace("_", " "): d.loc[d[r] == 1, "a_letto"].mean() for r in reperti})
ax = quota.plot.barh(figsize=(8, 3.5), color=["#9a9892", "#2a78d6", "#2a78d6", "#2a78d6", "#D4A13A"])
for i, v in enumerate(quota):
    ax.text(v + 0.01, i, f"{v:.0%}", va="center")
ax.set_xlim(0, 1.05); ax.set_xlabel("quota di radiografie fatte a letto")
ax.set_title("La proiezione tradisce il reperto"); plt.tight_layout(); plt.show()

### Passo 3 · Un modello cerca l'edema senza vedere l'immagine

Il modello impara su 7 pazienti su 10 e viene verificato sugli altri 3: nessun paziente sta in entrambe le parti.
Prima la regola più semplice possibile, «edema se la lastra è fatta a letto»; poi un modello con tutti i metadati.

**Atteso:** AUROC 0,756 con la sola proiezione, 0,787 con età, sesso e apparecchio.

In [ ]:
imp, ver = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0).split(d, groups=d.paziente))
d_imp, d_ver = d.iloc[imp], d.iloc[ver]
print(f"verifica: {len(d_ver):_} radiografie, {d_ver.edema.sum()} con edema".replace("_", "."))
print(f"solo la proiezione                     AUROC {roc_auc_score(d_ver.edema, d_ver.a_letto):.3f}")

colonne = ["a_letto", "eta", "donna", "larghezza", "altezza", "pixel_x", "pixel_y"]
modello = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(d_imp[colonne], d_imp.edema)
print(f"proiezione, età, sesso e apparecchio   AUROC {roc_auc_score(d_ver.edema, modello.predict_proba(d_ver[colonne])[:, 1]):.3f}")

### Passo 4 · Il risultato per sesso ed età, con quanti pazienti per gruppo

**Atteso:** donne 0,77, uomini 0,74; sopra i 70 anni 0,69, ma su soli 34 casi di edema.

In [ ]:
gruppi = {"tutti": d_ver.eta >= 0, "donne": d_ver.donna == 1, "uomini": d_ver.donna == 0,
          "sotto i 30 anni": d_ver.eta < 30, "30-49 anni": d_ver.eta.between(30, 49),
          "50-69 anni": d_ver.eta.between(50, 69), "70 anni e oltre": d_ver.eta >= 70}
pd.DataFrame([(g, m.sum(), d_ver.edema[m].sum(), roc_auc_score(d_ver.edema[m], d_ver.a_letto[m]))
              for g, m in gruppi.items()],
             columns=["gruppo", "radiografie", "casi di edema", "AUROC della sola proiezione"]).round(2)

---

### Cosa abbiamo visto

1. **L'indizio sta nel modo in cui è stata fatta la lastra:** la lastra a letto porta con sé la gravità del paziente.
2. **La scorciatoia non vale uguale per tutti:** cambia fra sessi e fasce d'età.
3. **Con pochi casi un risultato per gruppo non si può giudicare:** va sempre letto accanto al numero di pazienti.

### Da chiedere in gara

> **Da quali ospedali, con quali apparecchi e protocolli? Su quanti ospedali è stato verificato?**